# Probabilidade de Manutenção por Intervalo (Heads Semanais)

## Contexto

O modelo prevê **consumo futuro incremental por semana** (heads).  
Esses valores incrementais são **acumulados** para obter o consumo total previsto até cada semana futura.

Exemplo de previsões incrementais:

- head_1 = 200 km  
- head_2 = 250 km  
- head_3 = 180 km  
- head_4 = 220 km  

O **consumo acumulado previsto** fica:

| Semana | Consumo acumulado previsto |
|------:|----------------------------|
| 1 | 200 |
| 2 | 450 |
| 3 | 630 |
| 4 | 850 |

Além disso, existe um **marco real de manutenção**, definido por km ou horas.

Exemplo:
- Km atual: 8 250 km  
- Marco de manutenção: 10 000 km  

Então o **residual real** é:

Δ_real = 10 000 − 8 250 = 1 750 km

A pergunta fundamental é:

> Em qual semana o veículo vai atingir esse residual?  
> E com qual probabilidade isso ocorre em cada intervalo?

---

## 1. O que significa “probabilidade por intervalo”

Cada head define um **intervalo acumulado de consumo**:

- Semana 1:  
  0 < km ≤ 200

- Semana 2:  
  200 < km ≤ 450

- Semana 3:  
  450 < km ≤ 630

- Semana 4:  
  630 < km ≤ 850

A **probabilidade por intervalo** responde:

> Qual a probabilidade de o marco de manutenção cair dentro deste intervalo específico?

Isso é diferente de:
- “Vai ocorrer até a semana k?” (probabilidade acumulada)

Aqui queremos:
- “Vai ocorrer nesta semana?”

---

## 2. Onde entra o erro do modelo

O modelo fornece uma previsão, mas o valor real é:

consumo_real = consumo_previsto + erro

Para cada semana k, você já mediu historicamente:

erro_k = consumo_real_k − consumo_previsto_k

Esses erros, obtidos em validação ou teste, formam uma **distribuição empírica de erro**.

Essa distribuição representa a **incerteza do futuro**  
e é dela que a probabilidade é derivada.

---

## 3. Definições formais

Consumo acumulado previsto:
Ĉ_k = consumo acumulado previsto até a semana k

Consumo acumulado real:
C_k = Ĉ_k + E_k

onde E_k é o erro histórico acumulado até a semana k.

---

## 4. Evento “manutenção ocorre na semana k”

A manutenção ocorre **na semana k** quando:

C_(k−1) < Δ_real ≤ C_k

Substituindo pelos termos previstos + erro:

Ĉ_(k−1) + E_(k−1) < Δ_real ≤ Ĉ_k + E_k

Essa é a definição exata do evento.

---

## 5. Forma prática de calcular a probabilidade

O cálculo é feito em duas etapas.

### Passo A — Probabilidade acumulada até cada semana

Para cada semana k:

P_≤k = P(C_k ≥ Δ_real)  
     = P(Ĉ_k + E_k ≥ Δ_real)  
     = P(E_k ≥ Δ_real − Ĉ_k)

Estimativa empírica:

P_≤k ≈ mean(erro_k ≥ Δ_real − Ĉ_k)

---

### Passo B — Probabilidade por intervalo (por semana)

A probabilidade de ocorrer **exatamente na semana k** é:

P_k = P_≤k − P_≤(k−1)

com:
P_≤0 = 0

---

## 6. Exemplo numérico completo

Dados:

Ĉ_1 = 200  
Ĉ_2 = 450  
Ĉ_3 = 630  
Ĉ_4 = 850  

Δ_real = 400 km

Erros históricos (exemplo ilustrativo):

E_1 = [−50, 0, 30, 80, 120]  
E_2 = [−100, 20, 60, 150, 300]  
E_3 = [−150, 50, 100, 250, 400]  
E_4 = [−200, 80, 150, 350, 600]

---

### Probabilidade acumulada

Semana 1  
Δ_real − Ĉ_1 = 400 − 200 = 200  
P_≤1 = 0%

Semana 2  
Δ_real − Ĉ_2 = 400 − 450 = −50  
P_≤2 = 100%

Semana 3  
P_≤3 = 100%

Semana 4  
P_≤4 = 100%

---

### Probabilidade por intervalo

| Semana | Probabilidade |
|------:|---------------|
| 1 | 0% |
| 2 | 100% |
| 3 | 0% |
| 4 | 0% |

Conclusão: a manutenção ocorre na **semana 2**.

---

## 7. Exemplo com residual alto

Δ_real = 1 750 km

Consumo máximo previsto em 4 semanas:
Ĉ_4 = 850 km

Como 850 < 1 750  
e mesmo os maiores erros históricos não cobrem esse gap:

P(E_4 ≥ 900) ≈ 0

Então:

| Semana | Probabilidade |
|------:|---------------|
| 1 | 0% |
| 2 | 0% |
| 3 | 0% |
| 4 | 0% |

Conclusão: a manutenção **não ocorre dentro do horizonte previsto**.

---

## 8. Uso prático no sistema

Decisão determinística:
k* = menor k tal que Ĉ_k ≥ Δ_real

Decisão probabilística:
k* = menor k tal que P_≤k ≥ threshold

Exemplos de threshold:
- 30% → conservador  
- 50% → neutro  
- 80% → agressivo  

In [3]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd

from pathlib import Path

from api.config.output_config import OutputConfig
from moviasai.forecasting.metrics import load_predictions

In [5]:
output_cfg = OutputConfig.from_yaml('../config/output_config.yaml')

In [8]:
target = 'km'
pred_km_test = load_predictions(output_cfg.predictions_path('km'), 'test')

pred_km_train = load_predictions(output_cfg.predictions_path('km'), 'val')


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\f0pi\\git\\apimovias\\logs\\training\\km\\predictions\\metadata_test.csv'

In [8]:
pred_km_test

,veiculo_id,head,real,predicted
0,667,0,1511.30,1195.743728
1,667,1,1457.10,1224.497531
2,667,2,1281.70,1236.028203
3,667,3,903.00,1247.874957
4,6696,0,360.25,333.622309
...,...,...,...,...
58595,10614,3,2481.60,2297.513339
58596,3397,0,2610.80,2168.155319
58597,3397,1,1777.90,2073.630722
58598,3397,2,4155.60,2082.987415


In [10]:
pred_km_test['predicted'].describe()

count    58600.000000
mean      1475.490767
std        811.676788
min          0.000000
25%        749.451556
50%       1572.004976
75%       2068.914571
max       4503.773066
Name: predicted, dtype: float64

In [12]:
head_days = [7, 14, 21, 28]
n_heads = len(head_days)
upper = 0.5
for k in range(2, n_heads + 1):
    limits = k * upper * head_days[k - 1]
    print(k, head_days[k - 1], limits)
    print(f"Limite para {k} heads: {limits}")

2 14 14.0
Limite para 2 heads: 14.0
3 21 31.5
Limite para 3 heads: 31.5
4 28 56.0
Limite para 4 heads: 56.0
